# BAIM 4120 · Class 02 – White Belt
# From Alteryx to Python: the TalkingData Events File

This notebook has **two sections**:

| Section | What it is | What you do |
|---|---|---|
| **🟡 Part 1 – STARTER** | Explanations + code cells marked `# TODO` | Replace every `None` / `...` with working code and write your answers in the 📝 cells |
| **🟢 Part 2 – SOLUTION** | The complete, working code with explanations | Use it to **check** your work *after* you try each part yourself |

The question IDs (**A1, B3, D5 …**) match the lab handout `Class 02 - In-Class Lab.md`.

### What we are building (the same workflow as `Class 2 - In class Exercise Solution.yxmd`)

| Alteryx tool | pandas equivalent | Lab part |
|---|---|---|
| Input Data | `pd.read_csv()` | A |
| Browse | `df.head()`, `df.info()`, `df.dtypes` | A |
| Select – change types | `astype()`, `pd.to_datetime()` | B |
| Select – drop / rename | `drop()`, `rename()` | C |
| Filter `[lon] = 0` | boolean mask `df[mask]`, `df[~mask]` | D |
| Sort `device_id` desc | `sort_values(ascending=False)` | D |
| Output Data | `to_csv()` | E |

> **The pattern: ETL.** **E**xtract the data (read it in) → **T**ransform it (fix, filter, sort) → **L**oad it (save it). Almost every workflow you build in this course follows this pattern.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
# 🟡 PART 1 — STARTER

**How to work:**
1. Run **one cell at a time**, top to bottom. Don't use *Run All* yet.
2. Wherever you see `# TODO`, replace the `None` or `...` with your code.
3. Answer the **📝** questions in the Markdown cells (double-click a cell to edit it).
4. Stuck for more than 3 minutes? Read the hint, then the README, then ask your neighbour. Look at the Solution **last**.

## Setup

This cell is done for you. It imports pandas and chooses which data file to use.

- `pathlib.Path` represents a file path, and `.exists()` checks whether the file is there.
- The notebook uses the full dataset (~3.25 million rows) if it's in this folder, either as `events.csv` or as the `events.csv.zip` from the GitHub repo. pandas can read a `.zip` holding one CSV directly, so you don't need to unzip it. If neither is there, it uses the 10,000-row small file.
- If your laptop runs out of memory, change the last line to `DATA_FILE = SMALL_FILE`.

In [3]:
import pandas as pd
from pathlib import Path

FULL_ZIP   = Path("/content/drive/MyDrive/Python/CSV/Class 02. White Belt events.csv.zip")   # the version in the GitHub repo
FULL_FILE = Path("//aaasss//")


# Use the full CSV if you unzipped it, else the zip (pandas reads .zip directly), else the small file
if FULL_FILE.exists():
    DATA_FILE = FULL_FILE
elif FULL_ZIP.exists():
    DATA_FILE = FULL_ZIP
else:
    DATA_FILE = SMALL_FILE
print("Using:", DATA_FILE)

Using: /content/drive/MyDrive/Python/CSV/Class 02. White Belt events.csv.zip


## Part A – Extract & Browse  *(Alteryx: Input Data + Browse)*

### Concept: reading a CSV
A CSV file is plain text: just characters separated by commas. It has **no information about types**, so a program reading it has to *guess* whether `"654"` is a number or a word.

In this lab we turn guessing **off** with `dtype=str`, so every column comes in as text. That's the same thing Alteryx does by default (`V_String`). Starting from "everything is text" lets you see exactly what changes when you set the types yourself in Part B.

In [4]:
# TODO A0: Read DATA_FILE into a DataFrame called `raw`, with every column as text.
# Hint: pd.read_csv(<path>, dtype=str)
raw = pd.read_csv(DATA_FILE, dtype=str)
raw.head(10)

,event_id,device_id,timestamp,longitude,latitude
0,1,29182687948017175,2016-05-01 00:55:25,121.38,31.24
1,2,-6401643145415154744,2016-05-01 00:54:12,103.65,30.97
2,3,-4833982096941402721,2016-05-01 00:08:05,106.60,29.70
3,4,-6815121365017318426,2016-05-01 00:06:40,104.27,23.28
4,5,-5373797595892518570,2016-05-01 00:07:18,115.88,28.66
5,6,1476664663289716375,2016-05-01 00:27:21,0.00,0.00
6,7,5990807147117726237,2016-05-01 00:15:13,113.73,23.00
7,8,1782450055857303792,2016-05-01 00:15:35,113.94,34.70
8,9,-2073340001552902943,2016-05-01 00:15:33,0.00,0.00
9,10,-8195816569128397698,2016-05-01 00:41:31,119.34,26.04


### A1 – How many rows and columns?
A DataFrame's `.shape` is a **tuple** `(rows, columns)`. You can *unpack* it into two variables at once: `a, b = (1, 2)`.

In [5]:
# TODO A1: unpack raw.shape into n_rows and n_cols
n_rows, n_cols = raw.shape

print(f"Rows: {n_rows:,}   Columns: {n_cols}")

Rows: 3,252,950   Columns: 5


### A2 – What type is each column?
`.dtypes` lists the type of every column. `.info()` shows the types plus the row count and memory use.
In pandas 2.x, text shows up as `object`. In pandas 3.x it shows up as `str`. Both mean **text**.

In [6]:
# TODO A2: show the dtype of every column, then call .info()
print(raw.dtypes, end="\n\n")
raw.info()

event_id     object
device_id    object
timestamp    object
longitude    object
latitude     object
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3252950 entries, 0 to 3252949
Data columns (total 5 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   event_id   object
 1   device_id  object
 2   timestamp  object
 3   longitude  object
 4   latitude   object
dtypes: object(5)
memory usage: 124.1+ MB


### A3 – Find rows where the location is `0, 0`
Because everything is text right now, you must compare to the **text** `"0.00"`, not the number `0`.

A comparison like `raw["longitude"] == "0.00"` gives you a column of `True`/`False` values (a **mask**). Putting a mask inside `raw[ ... ]` keeps only the `True` rows.

In [7]:
# TODO A3: show the first 5 rows where longitude is the text "0.00"
raw[raw["longitude"] == "0.00"].head()

,event_id,device_id,timestamp,longitude,latitude
5,6,1476664663289716375,2016-05-01 00:27:21,0.00,0.00
8,9,-2073340001552902943,2016-05-01 00:15:33,0.00,0.00
15,16,9070651185984875886,2016-05-01 00:06:06,0.00,0.00
18,19,9161556264947841692,2016-05-01 00:17:37,0.00,0.00
29,30,-6615032828577640034,2016-05-01 00:28:03,0.00,0.00


📝 **A3 answer:** What does a location of `0, 0` really mean in this data? *(Hint: where is 0°, 0° on a map?)*

> 0,0 reprents not a place but rather a place to store data that is not known so that we can have a single search for this data as well in 0,0 to clean it.

📝 **A4 answer:** Name two pandas commands that do the job of the Alteryx **Browse** tool.

> Haven't taken Alteryx class yet but when doing research it seems like the pd.read command can do a good amount when getting data in to browse and also manipulating dfs by doing df = df[''] == 0.

## Part B – Select: fixing data types  *(Alteryx: Select tool → Type column)*

### Concept: why types matter
The type decides **what operations make sense** and **how values compare**:

| Type | pandas | Alteryx | Use it for | Example |
|---|---|---|---|---|
| Integer | `int64` | Int64 | Whole numbers, IDs | `event_id` |
| Decimal | `float64` | Double | Measurements | `longitude` |
| Date-time | `datetime64` | DateTime | Points in time | `timestamp` |
| Text | `object` / `str` | V_String | Names, codes | — |

- `series.astype("int64")` converts a column to a new type.
- `pd.to_datetime(series)` reads text like `"2016-05-01 00:55:25"` and turns it into a real date-time, so you can pull out the day or the hour later.
- We work on a **copy** (`raw.copy()`) so `raw` stays untouched and we can compare before and after.

In [8]:
df = raw.copy()

# TODO B1: convert each column to the correct type
df["event_id"] = df["event_id"].astype("int64")
df["device_id"] = df["device_id"].astype("int64")   # -> int64  (values go up to ~9.2 quintillion, which still fits in int64)
df["timestamp"] = pd.to_datetime(df['timestamp'])  #seried
df["longitude"] = df["longitude"].astype("float64")   # -> float64
df["latitude"] = df['latitude'].astype("float64")  # -> float64

df.dtypes

,0
event_id,int64
device_id,int64
timestamp,datetime64[ns]
longitude,float64
latitude,float64


### B2 – How many zeros are in `longitude`?
**Trick:** in Python, `True` counts as 1 and `False` as 0. So `(mask).sum()` **counts** the `True` rows, and `(mask).mean()` gives the **fraction** of rows that are `True`.

In [14]:
# TODO B2: count the zeros in df["longitude"] and the percentage of all rows
zero_lon = (mask1 := (raw["longitude"] == "0.00")).sum()
pct_zero = (mask1).mean()

print(f"Zeros in longitude: {zero_lon:,} ({pct_zero:.1%})")

Zeros in longitude: 968,703 (29.8%)


### B3 ⭐ – The `device_id` trap
Find the largest `device_id` **two ways**: once from `raw` (text) and once from `df` (integer). Compare them.

In [10]:
# TODO B3: largest device_id as TEXT (from raw) and as an INTEGER (from df)
largest_text = raw['device_id'].max()
largest_int  = df['device_id'].max()

print("Largest as text:   ", largest_text)
print("Largest as integer:", largest_int)

Largest as text:    99976251796408156
Largest as integer: 9222539910510672930


📝 **B3 answer:** Why does the text version give a different answer? What could go wrong in a real report?

> The text version just compares first number to second so 99 is bigger than 92, but doesnt take into account length of string as the 92 value is obviously a longer number, and the data reported could be understated to whoever reads the report.

📝 **B4 answer:** `device_id` has negative values. Should you ever calculate the *average* `device_id`? Why or why not?

> No, the device_id is just an index for devices by probably ip or serial number and the average of them would equate close to 0 and would have no statistical meaning in any context.

## Part C – Select: dropping & renaming columns  *(Alteryx: Select tool → uncheck / Rename)*

🔄 **Switch driver and navigator now.**

### Concept: pandas usually returns a *new* DataFrame
`df.drop(columns=[...])` and `df.rename(columns={...})` **don't change `df`**. They hand back a changed *copy*. To keep the change, assign the result to a variable: `new_df = df.drop(...)`.

In [11]:
# TODO C1: drop the timestamp column into a NEW DataFrame called df_no_time
df_no_time = df.drop(columns=['timestamp'])

# TODO C1: PROVE the column is gone (should print False)
print("timestamp still present?", "timestamp" in df_no_time.columns)

timestamp still present? False


In [34]:
# TODO C2: print the number of rows in df_no_time and compare with n_rows
print(f"Rows before: {n_rows:,}, Rows after: {len(df_no_time):,} Columns after: {df_no_time.shape[1]}")

Rows before: 3,252,950, Rows after: 3,252,950 Columns after: 4


In [13]:
# TODO C (rename): rename longitude -> lon and latitude -> lat, keep them as float64.
# Start from df (not df_no_time) so we keep the timestamp for later parts.
# Hint: df.rename(columns={"old": "new", ...})
df_clean = df.rename(columns={'longitude':'lon', 'latitude': "lat"})

df_clean.dtypes

,0
event_id,int64
device_id,int64
timestamp,datetime64[ns]
lon,float64
lat,float64


📝 **C2 answer:** Did the row count change when you dropped a column? Why?

> No, since the rows number is agnostic to the amount of columns in a df the dropping of a column only drops the data in that row for that column no the row itself.

📝 **C3 answer:** After `df.drop(...)`, why does `df` still have `timestamp`? What would you write to remove it from `df` itself?

> We would have to set df = to that permutation change in the data set but since we are attributing the drop to a new alternative df it does not write the change to df but the new df variable we named.

## Part D – Filter & Sort  *(Alteryx: Filter `[lon] = 0` and Sort `device_id` desc)*

### Concept: boolean masks = the Alteryx Filter tool
The Alteryx Filter tool has two outputs, **T** and **F**. In pandas:

```python
mask = df_clean["lon"] == 0      # True/False for every row
df_clean[mask]                    # T output: rows where mask is True
df_clean[~mask]                   # F output: ~ flips True <-> False
```

To **combine** conditions, use `&` (and) and `|` (or). Always put each condition in parentheses: `(a == 1) & (b != 2)`. Without them, Python's order of operations gives you an error or a wrong result.

In [15]:
# TODO D1: build the mask, then the T and F outputs
mask = df_clean["lon"] == 0
filter_true  = df_clean[mask]
filter_false = df_clean[~mask]

print(f"T (lon = 0):  {len(filter_true):,}")
print(f"F (lon != 0): {len(filter_false):,}")
# TODO D1: prove T + F equals the total number of rows (should print True)
print("Does T and F equal full length?", len(filter_true) + len(filter_false) == len(df_clean))

T (lon = 0):  968,711
F (lon != 0): 2,284,239
Does T and F equal full length? True


In [17]:
# TODO D3: count rows where lon == 0 BUT lat != 0
odd_rows = ((df_clean["lon"] == 0) & (df_clean['lat'] != 0)).sum()
print("lon == 0 but lat != 0:", odd_rows)

lon == 0 but lat != 0: 36


### Sorting
`sort_values("col", ascending=False)` puts the largest values first. `.head(n)` then shows the top `n` rows.
For the smallest values, sort ascending, or use the shortcut `df.nsmallest(n, "col")`.

In [23]:
# TODO D4: sort df_clean by device_id, largest first, into sorted_df
sorted_df = df_clean.sort_values('device_id', ascending = False)
sorted_df.head(10)

,event_id,device_id,timestamp,lon,lat
464435,464436,9222539910510672930,2016-05-05 15:40:36,0.00,0.00
2530268,2530269,9222355582733155698,2016-05-02 18:43:26,116.42,39.78
1097006,1097007,9222355582733155698,2016-05-02 18:37:01,116.42,39.78
3244331,3244332,9222355582733155698,2016-05-01 19:20:22,116.42,39.78
1859264,1859265,9222355582733155698,2016-05-02 18:39:10,116.42,39.78
2672071,2672072,9222355582733155698,2016-05-02 18:44:27,116.42,39.78
1183554,1183555,9222355582733155698,2016-05-02 18:47:53,116.42,39.78
1616246,1616247,9222355582733155698,2016-05-02 18:49:08,116.42,39.78
3163389,3163390,9222355582733155698,2016-05-02 18:47:27,116.42,39.78
526721,526722,9222355582733155698,2016-05-01 19:29:57,116.42,39.78


In [26]:
from numpy import negative
# TODO D5: show the 5 smallest (most negative) device_id values
n_sorted_df = df_clean.sort_values('device_id', ascending = True)
n_sorted_df.head(10)

,event_id,device_id,timestamp,lon,lat
420593,420594,-9222956879900151005,2016-05-06 15:38:31,113.24,23.19
379328,379329,-9222956879900151005,2016-05-06 15:40:43,113.24,23.19
205470,205471,-9222956879900151005,2016-05-06 15:36:46,113.24,23.19
514595,514596,-9222956879900151005,2016-05-07 12:17:29,113.24,23.19
2425796,2425797,-9222956879900151005,2016-05-07 11:54:59,113.24,23.19
2084995,2084996,-9222956879900151005,2016-05-06 15:34:42,0.00,0.00
2085016,2085017,-9222956879900151005,2016-05-06 15:41:40,113.24,23.19
2085014,2085015,-9222956879900151005,2016-05-06 15:33:24,113.24,23.19
926369,926370,-9222956879900151005,2016-05-07 11:45:15,113.24,23.19
289563,289564,-9222956879900151005,2016-05-06 21:26:44,113.24,23.19


📝 **D2 answer:** What does `~` do in `df_clean[~mask]`?

> It gives us the opposite answers of the mask, so it would be helpful for finding all of the values that are True in the mask and not false.

📝 **D4 answer:** The top row of `sorted_df` matches which answer from B3: the text one or the integer one?

> It matches the integer because the device_id is sorted by greatest to smallest and the values in the table we are using for id are ints and not strings so they sort correctly.

## Part E – Load & extend  *(Alteryx: Output Data)*

### Concept: saving with `to_csv`
`df.to_csv("file.csv", index=False)` writes a DataFrame to a CSV file.
The **index** is pandas' row label (0, 1, 2 …). With `index=False` it isn't written to the file. Without it, you get an extra unnamed column of row numbers.

⚠️ On the full dataset, these files are 100–200 MB each.

In [28]:
# TODO E (save): save sorted_df to "events_clean.csv" WITHOUT the index
sorted_df.to_csv('sorted.data_white.belt.csv', index=False)

In [30]:
# TODO E1: experiment! Save sorted_df.head(5) twice, with index=True and with index=False,
# then read each one back with pd.read_csv and compare the columns.
sorted_df.head(5).to_csv("w_index.wb.csv", index=True)
sorted_df.head(5).to_csv("w.o_index.wb.csv", index=False)

print("With index:   ", list(pd.read_csv("w_index.wb.csv").columns))
print("Without index:   ", list(pd.read_csv("w.o_index.wb.csv").columns))

With index:    ['Unnamed: 0', 'event_id', 'device_id', 'timestamp', 'lon', 'lat']
Without index:    ['event_id', 'device_id', 'timestamp', 'lon', 'lat']


In [31]:
# TODO E2: save only the F output (real locations) to "events_real_locations.csv"
# and print how many rows you wrote
filter_false.to_csv('events_real_loactions.csv', index = False)
print(f"Saved {len(filter_false):,} rows to events_real_locations.csv")

Saved 2,284,239 rows to events_real_locations.csv


In [32]:
# TODO E3: how many UNIQUE devices are there?   Hint: .nunique()
n_devices = df_clean["device_id"].nunique()
print(f"Unique devices: {n_devices:,}")

Unique devices: 60,865


### E4 – Busiest day
A datetime column has a `.dt` **accessor** that gets at the date parts: `.dt.date`, `.dt.hour`, `.dt.day_name()` …
`value_counts()` counts how often each value appears and sorts the result, most common first.

In [33]:
# TODO E4: which calendar day has the most events, and how many?
day_counts = df_clean['timestamp'].dt.date.value_counts()
print(f'Busiest day: {day_counts.index[0]} with {day_counts.iloc[0]:,} events\n')
day_counts.head()

Busiest day: 2016-05-03 with 483,293 events



,count
timestamp,
2016-05-03,483293
2016-05-05,478999
2016-05-04,473487
2016-05-06,471730
2016-05-02,451546


📝 **E1 answer:** What does `index=False` do? What happens if you leave it out?

> The csv returns an index row with it, which would be redundant becuase if we brought it back to work with an index would be auto generated for us in that instance.

---
## 📝 Reflection (3–4 sentences)
Compare doing this in Alteryx vs. Python. What was easier in each? When would you pick one over the other?

> Again, I have not taken the Alteryx class but I would have to imagine that this coding is easier in execution but harder in premise than Alteryx since the UX is trying to be more user friendly than just a console. So, since I have learned python before Alt. I would like to continue using python as it will have less bounds for me as I continue learning with it!

---
---
# 🟢 PART 2 — SOLUTION

> **Try the Starter first!** You learn far more by getting stuck and getting yourself unstuck than by reading finished code.

This section runs on its own: it reloads the data from scratch, so you can run it without finishing the Starter.

In [ ]:
import pandas as pd
from pathlib import Path

FULL_FILE  = Path("Class 02. White Belt events.csv")
FULL_ZIP   = Path("Class 02. White Belt events.csv.zip")   # the version in the GitHub repo
SMALL_FILE = Path("Class 02. White Belt events_small-1.csv")

# Use the full CSV if you unzipped it, else the zip (pandas reads .zip directly), else the small file
if FULL_FILE.exists():
    DATA_FILE = FULL_FILE
elif FULL_ZIP.exists():
    DATA_FILE = FULL_ZIP
else:
    DATA_FILE = SMALL_FILE
print("Using:", DATA_FILE)

Using: Class 02. White Belt events.csv


## Part A – Extract & Browse (solution)

In [ ]:
# A0: read every column as text (same as Alteryx's V_String default)
raw = pd.read_csv(DATA_FILE, dtype=str)
raw.head(10)

,event_id,device_id,timestamp,longitude,latitude
0,1,29182687948017175,2016-05-01 00:55:25,121.38,31.24
1,2,-6401643145415154744,2016-05-01 00:54:12,103.65,30.97
2,3,-4833982096941402721,2016-05-01 00:08:05,106.60,29.70
3,4,-6815121365017318426,2016-05-01 00:06:40,104.27,23.28
4,5,-5373797595892518570,2016-05-01 00:07:18,115.88,28.66
5,6,1476664663289716375,2016-05-01 00:27:21,0.00,0.00
6,7,5990807147117726237,2016-05-01 00:15:13,113.73,23.00
7,8,1782450055857303792,2016-05-01 00:15:35,113.94,34.70
8,9,-2073340001552902943,2016-05-01 00:15:33,0.00,0.00
9,10,-8195816569128397698,2016-05-01 00:41:31,119.34,26.04


In [ ]:
# A1: .shape is a (rows, columns) tuple; unpack it into two variables
n_rows, n_cols = raw.shape
print(f"Rows: {n_rows:,}   Columns: {n_cols}  ->  {list(raw.columns)}")

Rows: 3,252,950   Columns: 5  ->  ['event_id', 'device_id', 'timestamp', 'longitude', 'latitude']


In [ ]:
# A2: every column is text before we convert anything
print(raw.dtypes, end="\n\n")
raw.info()

event_id     str
device_id    str
timestamp    str
longitude    str
latitude     str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 3252950 entries, 0 to 3252949
Data columns (total 5 columns):
 #   Column     Dtype
---  ------     -----
 0   event_id   str  
 1   device_id  str  
 2   timestamp  str  
 3   longitude  str  
 4   latitude   str  
dtypes: str(5)
memory usage: 124.1 MB


In [ ]:
# A3: compare to the TEXT "0.00", because the column is still text
raw[raw["longitude"] == "0.00"].head()

,event_id,device_id,timestamp,longitude,latitude
5,6,1476664663289716375,2016-05-01 00:27:21,0.00,0.00
8,9,-2073340001552902943,2016-05-01 00:15:33,0.00,0.00
15,16,9070651185984875886,2016-05-01 00:06:06,0.00,0.00
18,19,9161556264947841692,2016-05-01 00:17:37,0.00,0.00
29,30,-6615032828577640034,2016-05-01 00:28:03,0.00,0.00


**A3:** `0°, 0°` is in the Atlantic Ocean off West Africa ("Null Island"). Nobody is using a phone app there. A `0, 0` location is a **placeholder for "location unknown"**: the phone didn't share its GPS position. Treat these rows as **missing** location data, not real places.

**A4:** `raw.head()` / `raw.head(10)` shows the rows (like Browse's data view). `raw.info()` and `raw.dtypes` show the columns and types (like Browse's metadata/profile view). `raw.describe()` also works.

## Part B – Fixing data types (solution)

In [ ]:
df = raw.copy()     # keep raw untouched so we can compare before/after

df["event_id"]  = df["event_id"].astype("int64")
df["device_id"] = df["device_id"].astype("int64")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["longitude"] = df["longitude"].astype("float64")
df["latitude"]  = df["latitude"].astype("float64")

df.dtypes

event_id              int64
device_id             int64
timestamp    datetime64[us]
longitude           float64
latitude            float64
dtype: object

**B1:**

| Column | Before | After | Function |
|---|---|---|---|
| `event_id` | text | `int64` | `.astype("int64")` |
| `device_id` | text | `int64` | `.astype("int64")` |
| `timestamp` | text | `datetime64` | `pd.to_datetime()` |
| `longitude` | text | `float64` | `.astype("float64")` |
| `latitude` | text | `float64` | `.astype("float64")` |

In [ ]:
# B2: True counts as 1, so .sum() counts matches and .mean() gives the fraction
zero_lon = (df["longitude"] == 0).sum()
pct_zero = (df["longitude"] == 0).mean()
print(f"Zeros in longitude: {zero_lon:,} ({pct_zero:.1%})")

Zeros in longitude: 968,711 (29.8%)


In [ ]:
# B3: the same "max" question, asked of text vs. numbers
largest_text = raw["device_id"].max()   # alphabetical (WRONG for numbers)
largest_int  = df["device_id"].max()    # numeric (CORRECT)
print("Largest as text:   ", largest_text)
print("Largest as integer:", largest_int)

Largest as text:    99976251796408156
Largest as integer: 9222539910510672930


**B3 – why the text answer is wrong:** text is compared **one character at a time, left to right**, like words in a dictionary. `"99976…"` and `"92225…"` both start with `9`. At the second character, `9` beats `2`, so the text comparison stops there and calls `"99976…"` bigger. It never notices that the other number has **two more digits**. Numbers are compared by **value**, so `9,222,539,910,510,672,930` correctly wins.

*Business impact:* "top 10 customers", "highest invoice", or "latest order number" reports would quietly show the wrong records. Nothing crashes, so nobody notices. **Set types before you sort, compare, or calculate.**

**B4:** No. `device_id` is an **identifier** (a label), not a **quantity**. An average of IDs has no meaning, like averaging phone numbers. The negative values are just part of how the IDs were generated. A good habit: ask *"does arithmetic on this column mean anything?"* If not, it's an ID or a category, even if it looks like a number.

## Part C – Dropping & renaming (solution)

In [ ]:
# C1: drop returns a NEW DataFrame; df itself is unchanged
df_no_time = df.drop(columns=["timestamp"])
print("timestamp still present?", "timestamp" in df_no_time.columns)   # False = proof
print("df still has it?        ", "timestamp" in df.columns)           # True -> see C3
df_no_time.head()

timestamp still present? False
df still has it?         True


,event_id,device_id,longitude,latitude
0,1,29182687948017175,121.38,31.24
1,2,-6401643145415154744,103.65,30.97
2,3,-4833982096941402721,106.60,29.70
3,4,-6815121365017318426,104.27,23.28
4,5,-5373797595892518570,115.88,28.66


In [ ]:
# C2: dropping a COLUMN never removes ROWS
print(f"Rows before: {n_rows:,}   Rows after: {len(df_no_time):,}   Columns after: {df_no_time.shape[1]}")

Rows before: 3,252,950   Rows after: 3,252,950   Columns after: 4


In [ ]:
# Rename and confirm the float type (pandas float64 = Alteryx Double; float32 = Alteryx Float)
df_clean = (
    df.rename(columns={"longitude": "lon", "latitude": "lat"})
      .astype({"lon": "float64", "lat": "float64"})
)
df_clean.dtypes

event_id              int64
device_id             int64
timestamp    datetime64[us]
lon                 float64
lat                 float64
dtype: object

**C2:** No. `drop(columns=...)` removes a column (it shrinks the table **sideways**). Every row is still there, just with one fewer field.

**C3:** `drop()` returns a **new** DataFrame and leaves the original alone. That's on purpose: you can't wreck your data by accident. To change `df` itself, reassign it: `df = df.drop(columns=["timestamp"])`. (You may see `inplace=True` online. The pandas team discourages it, so prefer reassigning.)

*Style note:* the rename cell chains two methods inside parentheses: `df.rename(...).astype(...)`. Each method returns a DataFrame, so the next one can be called on it. The parentheses let you split the chain over several lines.

## Part D – Filter & Sort (solution)

In [ ]:
# D1: one mask, two outputs, just like Alteryx's T and F anchors
mask = df_clean["lon"] == 0
filter_true  = df_clean[mask]
filter_false = df_clean[~mask]

print(f"T (lon = 0):  {len(filter_true):,}")
print(f"F (lon != 0): {len(filter_false):,}")
print("T + F == total?", len(filter_true) + len(filter_false) == len(df_clean))

T (lon = 0):  968,711
F (lon != 0): 2,284,239
T + F == total? True


In [ ]:
# D3: combine conditions with &, each in its own parentheses
odd_rows = ((df_clean["lon"] == 0) & (df_clean["lat"] != 0)).sum()
print("lon == 0 but lat != 0:", odd_rows)
df_clean[(df_clean["lon"] == 0) & (df_clean["lat"] != 0)].head()

lon == 0 but lat != 0: 36


,event_id,device_id,timestamp,lon,lat
76903,76904,-207934154262067872,2016-05-04 09:53:05,0.0,-0.37
143102,143103,8124653877439841139,2016-05-01 11:21:17,0.0,0.02
285831,285832,8508167634391869020,2016-05-06 18:19:12,-0.0,-0.26
313831,313832,8124653877439841139,2016-05-04 17:00:51,0.0,0.02
393345,393346,8508167634391869020,2016-05-06 18:16:53,-0.0,-0.26


**D3 – why do these rows exist?** A handful of events have `lon = 0` with a real-looking latitude. That's most likely a **data-quality glitch**: a partly recorded GPS fix, or a bug in the app that logged it. Treat these as **unreliable** locations too. Real datasets always have small oddities like this, and finding them is part of the job.

**Why the parentheses matter:** in Python, `&` is evaluated *before* `==`. So `df["lon"] == 0 & df["lat"] != 0` is read as `df["lon"] == (0 & df["lat"]) != 0`, which gives a wrong result or an error.

In [ ]:
# D4: largest first
sorted_df = df_clean.sort_values("device_id", ascending=False)
sorted_df.head(10)

,event_id,device_id,timestamp,lon,lat
464435,464436,9222539910510672930,2016-05-05 15:40:36,0.00,0.00
2918963,2918964,9222355582733155698,2016-05-01 19:22:52,116.42,39.78
1234069,1234070,9222355582733155698,2016-05-02 18:40:08,116.42,39.78
1911478,1911479,9222355582733155698,2016-05-01 19:27:09,116.42,39.78
1071394,1071395,9222355582733155698,2016-05-01 19:21:23,116.42,39.78
1616246,1616247,9222355582733155698,2016-05-02 18:49:08,116.42,39.78
3163389,3163390,9222355582733155698,2016-05-02 18:47:27,116.42,39.78
558308,558309,9222355582733155698,2016-05-01 19:26:41,116.42,39.78
1546120,1546121,9222355582733155698,2016-05-01 19:19:54,116.42,39.78
1403595,1403596,9222355582733155698,2016-05-01 19:21:52,116.42,39.78


In [ ]:
# D5: the 5 smallest device_ids (nsmallest is a shortcut for sort ascending + head)
# One device has many events, so the 5 smallest ROWS can repeat the same ID:
display(df_clean.nsmallest(5, "device_id"))

# For the 5 smallest DISTINCT IDs, remove duplicates first:
df_clean["device_id"].drop_duplicates().nsmallest(5)

,event_id,device_id,timestamp,lon,lat
11280,11281,-9222956879900151005,2016-05-07 11:36:04,0.00,0.00
12678,12679,-9222956879900151005,2016-05-07 12:18:35,113.24,23.19
49382,49383,-9222956879900151005,2016-05-07 15:44:45,0.00,0.00
131848,131849,-9222956879900151005,2016-05-07 12:06:35,113.24,23.19
205470,205471,-9222956879900151005,2016-05-06 15:36:46,113.24,23.19


11280    -9222956879900151005
118331   -9222661944218806987
277134   -9222399302879214035
74538    -9221825537663503111
269592   -9221767098072603291
Name: device_id, dtype: int64

**D2:** `~` is **NOT** for a boolean Series: it flips every `True` to `False` and every `False` to `True`. So `df_clean[~mask]` keeps the rows where `lon` is **not** 0 (Alteryx's **F** output).

**D5:** One device logs many events, so `nsmallest(5, "device_id")` can show the **same ID several times**. Use `drop_duplicates()` first if you want 5 *different* devices. Both answers are accepted; just say which one you gave.

**D4:** The top row matches the **integer** answer from B3 (`9222539910510672930`), because `device_id` is now `int64` and sorts by value. Notice that its location is `0, 0`: this device's events are also in the T output.

## Part E – Load & extend (solution)

In [ ]:
# Save the cleaned, sorted data (the L in ETL)
sorted_df.to_csv("events_clean.csv", index=False)
print(f"Saved {len(sorted_df):,} rows to events_clean.csv")

Saved 3,252,950 rows to events_clean.csv


In [ ]:
# E1: see what index=True vs index=False does
sorted_df.head(5).to_csv("demo_with_index.csv", index=True)
sorted_df.head(5).to_csv("demo_no_index.csv",   index=False)

print("With index:   ", list(pd.read_csv("demo_with_index.csv").columns))
print("Without index:", list(pd.read_csv("demo_no_index.csv").columns))

With index:    ['Unnamed: 0', 'event_id', 'device_id', 'timestamp', 'lon', 'lat']
Without index: ['event_id', 'device_id', 'timestamp', 'lon', 'lat']


**E1:** With the default `index=True`, pandas writes its row labels as an extra first column. When you read the file back, that column shows up as **`Unnamed: 0`**: junk that piles up each time you save and reload. `index=False` leaves it out, so the file has only your real columns. *(Alteryx's Output Data tool never writes an index, so `index=False` matches Alteryx.)*

In [ ]:
# E2: save only the rows with a real location (the F output)
filter_false.to_csv("events_real_locations.csv", index=False)
print(f"Saved {len(filter_false):,} rows to events_real_locations.csv")

Saved 2,284,239 rows to events_real_locations.csv


In [ ]:
# E3: nunique counts distinct values
n_devices = df_clean["device_id"].nunique()
print(f"Unique devices: {n_devices:,}")
print(f"Average events per device: {len(df_clean) / n_devices:.1f}")

Unique devices: 60,865
Average events per device: 53.4


In [ ]:
# E4: .dt.date takes the calendar day from each timestamp; value_counts sorts most-common first
day_counts = df_clean["timestamp"].dt.date.value_counts()
print(f"Busiest day: {day_counts.index[0]} with {day_counts.iloc[0]:,} events\n")
day_counts.sort_index().to_frame("events")

Busiest day: 2016-05-03 with 483,293 events



,events
timestamp,
2016-04-30,959
2016-05-01,444589
2016-05-02,451546
2016-05-03,483293
2016-05-04,473487
2016-05-05,478999
2016-05-06,471730
2016-05-07,448345
2016-05-08,2


**Reflection (sample answer):** Alteryx makes the flow easy to *see*. You can watch the data move from tool to tool, and the T/F outputs of the Filter tool are clear. Python takes more typing, but every step is written down, so it's easy to repeat, share, version in Git, and extend (for example, the E4 busiest-day question was one extra line). Pick Alteryx for quick visual exploration and for handing workflows to non-programmers. Pick Python when you need automation, custom logic, or free tools.

## Summary of answers

In [ ]:
pd.Series({
    "A1  Rows": f"{n_rows:,}",
    "A1  Columns": n_cols,
    "B2  Zeros in longitude": f"{zero_lon:,} ({pct_zero:.1%})",
    "B3  Largest device_id (text, WRONG)": largest_text,
    "B3  Largest device_id (integer)": largest_int,
    "C2  Rows after dropping timestamp": f"{len(df_no_time):,}",
    "D1  T output (lon = 0)": f"{len(filter_true):,}",
    "D1  F output (lon != 0)": f"{len(filter_false):,}",
    "D3  lon = 0 but lat != 0": odd_rows,
    "E3  Unique devices": f"{n_devices:,}",
    "E4  Busiest day": f"{day_counts.index[0]} ({day_counts.iloc[0]:,} events)",
}).to_frame("Answer")

,Answer
A1 Rows,"3,252,950"
A1 Columns,5
B2 Zeros in longitude,"968,711 (29.8%)"
"B3 Largest device_id (text, WRONG)",99976251796408156
B3 Largest device_id (integer),9222539910510672930
C2 Rows after dropping timestamp,"3,252,950"
D1 T output (lon = 0),"968,711"
D1 F output (lon != 0),"2,284,239"
D3 lon = 0 but lat != 0,36
E3 Unique devices,"60,865"
